In [3]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np

from pinn_loader import CaseData


INPUT_DIM = 4
OUTPUT_DIM = 3


@dataclass
class SamplerConfig:
    seed: int = 123

    n_data: int = 8192
    n_pde: int = 8192
    n_bc_inlet: int = 2048
    n_bc_outlet: int = 2048
    n_bc_wall: int = 2048
    n_bc_body: int = 2048
    n_extra: int = 2048

    mix_mode: str = "balanced"
    pde_from: str = "field"
    verbose: bool = True


@dataclass
class Batch:
    X_data: np.ndarray
    Y_data: np.ndarray
    has_data: bool

    X_pde: np.ndarray
    bc: Dict[str, Dict[str, np.ndarray]]

    case_id: str
    shape: str
    Re: int

    data_used: str
    targets: Dict[str, float]

    extra: Optional[Dict[str, np.ndarray]]
    probes: Optional[Dict[str, np.ndarray]]


def _choice(rng: np.random.Generator, N: int, k: int) -> np.ndarray:
    if N <= 0 or k <= 0:
        return np.zeros((0,), dtype=np.int64)
    if k >= N:
        return rng.integers(0, N, size=k, dtype=np.int64)
    return rng.choice(N, size=k, replace=False)


def _empty_X(n: int = 0) -> np.ndarray:
    return np.zeros((n, INPUT_DIM), dtype=np.float32)


def _empty_Y(n: int = 0) -> np.ndarray:
    return np.zeros((n, OUTPUT_DIM), dtype=np.float32)


class MultiCaseSampler:
    def __init__(self, cases: List[CaseData], cfg: SamplerConfig):
        self.cases = cases
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)

        if self.cfg.mix_mode not in {"balanced", "proportional"}:
            raise ValueError("cfg.mix_mode must be 'balanced' or 'proportional'")
        if self.cfg.pde_from != "field":
            raise ValueError("cfg.pde_from currently supports only 'field'")

        self.valid_case_indices = [
            i for i, c in enumerate(self.cases)
            if self._case_has_field(c)
        ]

        if len(self.valid_case_indices) == 0:
            self.weights = np.zeros((0,), dtype=np.float64)
        else:
            if self.cfg.mix_mode == "proportional":
                ns = np.array(
                    [self.cases[i].field_X.shape[0] for i in self.valid_case_indices],
                    dtype=np.float64,
                )
                s = ns.sum()
                self.weights = ns / s if s > 0 else np.ones(len(ns), dtype=np.float64) / len(ns)
            else:
                self.weights = np.ones(len(self.valid_case_indices), dtype=np.float64) / len(self.valid_case_indices)

    @staticmethod
    def _case_has_field(c: CaseData) -> bool:
        return c.field_X is not None and c.field_X.ndim == 2 and c.field_X.shape[0] > 0 and c.field_X.shape[1] == INPUT_DIM

    @staticmethod
    def _case_has_field_labels(c: CaseData) -> bool:
        if not MultiCaseSampler._case_has_field(c):
            return False
        if c.field_Y is None or c.field_Y.ndim != 2:
            return False
        if c.field_Y.shape[0] != c.field_X.shape[0]:
            return False
        if c.field_Y.shape[1] != OUTPUT_DIM:
            return False
        return bool(c.has_data)

    @staticmethod
    def _case_has_bc(c: CaseData, bc_type: str) -> bool:
        if bc_type not in c.bc:
            return False
        X = c.bc[bc_type].get("X", None)
        if X is None or X.ndim != 2:
            return False
        return X.shape[0] > 0 and X.shape[1] == INPUT_DIM

    @staticmethod
    def _compose_case_data_used(c: CaseData) -> str:
        parts: List[str] = []

        if MultiCaseSampler._case_has_field(c):
            parts.append("pde_field")
        if MultiCaseSampler._case_has_field_labels(c):
            parts.append("supervised_uvp")

        if c.extra is not None and c.extra["X"].shape[0] > 0:
            parts.append("extra:dpdx,dpdy")
        if c.probes is not None:
            parts.append("probes:front,rear")

        bc_parts = []
        for bc_type in ["inlet", "outlet", "wall", "body"]:
            if MultiCaseSampler._case_has_bc(c, bc_type):
                bc_parts.append(bc_type)
        if bc_parts:
            parts.append("bc:" + ",".join(bc_parts))

        return " | ".join(parts) if parts else "none"

    def _pick_case(self) -> CaseData:
        if len(self.valid_case_indices) == 0:
            raise RuntimeError("No valid cases available.")
        j = int(self.rng.choice(len(self.valid_case_indices), p=self.weights))
        return self.cases[self.valid_case_indices[j]]

    def _sample_supervised_data(self, c: CaseData):
        has_data = self._case_has_field_labels(c)
        if not has_data or self.cfg.n_data <= 0:
            return _empty_X(), _empty_Y(), False

        idx = _choice(self.rng, c.field_X.shape[0], self.cfg.n_data)
        if len(idx) == 0:
            return _empty_X(), _empty_Y(), False

        return c.field_X[idx], c.field_Y[idx], True

    def _sample_pde(self, c: CaseData) -> np.ndarray:
        if not self._case_has_field(c) or self.cfg.n_pde <= 0:
            return _empty_X()

        idx = _choice(self.rng, c.field_X.shape[0], self.cfg.n_pde)
        if len(idx) == 0:
            return _empty_X()

        return c.field_X[idx]

    def _sample_one_bc(self, c: CaseData, bc_type: str, n: int):
        if bc_type not in c.bc:
            return {"X": _empty_X(), "Y": _empty_Y(), "has_Y": False}

        X_all = c.bc[bc_type].get("X", _empty_X())
        Y_all = c.bc[bc_type].get("Y", _empty_Y())
        has_Y = bool(c.bc[bc_type].get("has_Y", False))

        if X_all is None or X_all.ndim != 2 or X_all.shape[0] == 0 or n <= 0:
            return {"X": _empty_X(), "Y": _empty_Y(), "has_Y": has_Y}

        idx = _choice(self.rng, X_all.shape[0], n)
        if len(idx) == 0:
            return {"X": _empty_X(), "Y": _empty_Y(), "has_Y": has_Y}

        Xb = X_all[idx]
        Yb = Y_all[idx] if (Y_all is not None and Y_all.ndim == 2 and Y_all.shape[0] == X_all.shape[0]) else _empty_Y(len(idx))
        return {"X": Xb, "Y": Yb, "has_Y": has_Y}

    def _sample_extra(self, c: CaseData) -> Optional[Dict[str, np.ndarray]]:
        if c.extra is None or self.cfg.n_extra <= 0:
            return None

        X_all = c.extra["X"]
        gx_all = c.extra["dpdx"]
        gy_all = c.extra["dpdy"]

        if X_all.ndim != 2 or X_all.shape[0] == 0:
            return None

        idx = _choice(self.rng, X_all.shape[0], self.cfg.n_extra)
        if len(idx) == 0:
            return None

        return {
            "X": X_all[idx],
            "dpdx": gx_all[idx],
            "dpdy": gy_all[idx],
        }

    def next_batch(self) -> Batch:
        c = self._pick_case()
        data_used = self._compose_case_data_used(c)

        X_data, Y_data, has_data = self._sample_supervised_data(c)
        X_pde = self._sample_pde(c)

        bc_out = {
            "inlet": self._sample_one_bc(c, "inlet", self.cfg.n_bc_inlet),
            "outlet": self._sample_one_bc(c, "outlet", self.cfg.n_bc_outlet),
            "wall": self._sample_one_bc(c, "wall", self.cfg.n_bc_wall),
            "body": self._sample_one_bc(c, "body", self.cfg.n_bc_body),
        }

        extra_out = self._sample_extra(c)
        probes_out = c.probes

        return Batch(
            X_data=X_data,
            Y_data=Y_data,
            has_data=bool(has_data),
            X_pde=X_pde,
            bc=bc_out,
            case_id=c.paths.case_id,
            shape=c.paths.shape,
            Re=int(c.paths.Re),
            data_used=data_used,
            targets=dict(c.targets),
            extra=extra_out,
            probes=probes_out,
        )